In [14]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import (
    GroupKFold, GridSearchCV, cross_val_score, KFold
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor





file_path_EMG = r"D:\ML project\guided\guided_dataset_X.npy"
dataEMG = np.load(file_path_EMG)

file_path_HAND = r"D:\ML project\guided\guided_dataset_y.npy"
dataHAND = np.load(file_path_HAND)


#We start by doing a filtering of our data. "https://www.sciencedirect.com/science/article/pii/S0021929010000631" this paper is a relevant litterature on the subject
#We learn that most of the noise that occurs when recording our sEMG is located in lower frequencies. And that a high-pass filtering at 10hz is highly beneficial with diminishing performance at 20 and 30 hz 
#while still being beneficial as not much revelant signal information is lost.
#As the choice of the frequencie of the high-pass is muscle dependent and as we are limited by our lack of expertise. We will follow the recommendation of the paper and settle at doing a Butterworth filter with a corner frequency of 20 Hz and a slope of 12 dB/oct 
#To do so we will use the butter and filtfilt functions of the scipy library

#In addition, we also learn from this paper "https://iopscience.iop.org/article/10.1088/1742-6596/1237/3/032008/pdf" that most of the sEMG's energy is concentrated in the 0-500hz range 
#Our sampling frenquencie being of 1024hz, twice the bandwith of the signal, by the Nyquist-Shannon theorem we can efficiently control for aliasing up to the 500hz frequencies.
#Even if according to the first paper most of the noise is located in lower frequencies, since relveant information is under the 500hz threshold, ça mange pas de pain to control for higher frequencies.


cutoff_high = 20       # We select (in hz) the cutoff of our high-pass filter
cutoff_low = 500     # We select (in hz) the cutoff of our low-pass filter
order_high = 2         # We select the number of poles. In the Butterworth filter, each pole correspong to 6db, so we select two to get the recommended 12dp/octave
order_low = 4        # Once again we select the number of poles. In the paper the nbr of decibel/octave for the high-pass filter was 24 dB/oct, so we follow this recommendation.

b_high, a_high = butter(order_high, cutoff_high / (1024 / 2), btype='highpass') #The butter function returns the filter coefficients a (denominator),b (numerator). And take as inupts the order and Wn, the desired cutoff frequency being the frequency at which at which the magnitude response of the filter is 1 / √2
b_low, a_low = butter(order_low, cutoff_low / (1024 / 2), btype='lowpass')

first_filtering = filtfilt(b_high, a_high, dataEMG)  #Here we use the filtfilt function, which controls for distortion. That filters forward the backward the forward then blabla, à compléter, pas encore parfaitement compris. https://dsp.stackexchange.com/questions/9467/what-is-the-advantage-of-matlabs-filtfilt 
final_filtering = filtfilt(b_low, a_low, first_filtering)

missingNO = np.isnan(final_filtering).sum()   #Here we simply check that our dataset doesn't have missing values. Or we would have to take that into account for our pipeline
print("Missing Values:", missingNO)   


n_sessions, n_electrodes, n_samples = final_filtering.shape


#Dois encore copier coller les sources.



Missing Values: 0


In [15]:
#We start by "windowing" our data set by creating overlapping windows of size 500ms with a 250ms step betwwen each windows
#this results in the creation of 919 session per electrode, 6433 per session and a total of 25732

ws = 500                # Size of our windows
step = 250              # Distance between the beginning of each windows 2 by 2
n_samples = 230000      # Total time of a session

#To do so we use a nested loop that will go over the data of each electrodes of each session and add it in a two level dictionnary
emg_windows = {}
hand_windows = {}
n_joints     = dataHAND.shape[1]   
n_samples    = dataEMG.shape[2]


for s in range(n_sessions):
    emg_windows[s] = {}
    hand_windows[s] = {}
    for e in range(n_electrodes):
        emg_windows[s][e] = []
        for start in range(0, n_samples - ws + 1, step):
            win = dataEMG[s, e, start : start + ws]
            emg_windows[s][e].append(win)



    for j in range(n_joints):
        hand_windows[s][j] = []
        for start in range(0, n_samples - ws + 1, step):
             win_hand = dataHAND[s, j, start : start + ws]
             hand_windows[s][j].append(win_hand)


    




In [38]:
from sklearn.model_selection import LeaveOneGroupOut


windows_list = []
hand_last    = []
groups = []


for s in range(5):
    for w in range(918):
        groups.append(s)

        
        emg_stack = np.stack([emg_windows[s][e][w]
                              for e in range(n_electrodes)],
                             axis=0)
        windows_list.append(emg_stack)


        last_samples = [hand_windows[s][j][w][-1]
            for j in range(51)]
        hand_last.append(last_samples)




X = np.stack(windows_list, axis=0) 
Y = np.array(hand_last)
groups = np.array(groups) 

print("X.shape =", X.shape)
print("Y.shape =", Y.shape)
print("groups.shape =", groups.shape)

print(groups)


logo = LeaveOneGroupOut()
logo.get_n_splits(X, Y, groups)
logo.get_n_splits(groups=groups)
print(logo)
for i, (train_index, test_index) in enumerate(logo.split(X, Y, groups)):
    print(f"Fold {i}:")

    print(f"  Train: index={train_index}, group={groups[train_index]}")

    print(f"  Test:  index={test_index}, group={groups[test_index]}")

X.shape = (4590, 8, 500)
Y.shape = (4590, 51)
groups.shape = (4590,)
[0 0 0 ... 4 4 4]
LeaveOneGroupOut()
Fold 0:
  Train: index=[ 918  919  920 ... 4587 4588 4589], group=[1 1 1 ... 4 4 4]
  Test:  index=[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 

In [32]:
from sklearn.base import BaseEstimator
from scipy.stats    import skew, kurtosis
from scipy.signal   import find_peaks
from numpy.fft      import rfft, rfftfreq

class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.sigma = 0.05
        self.features_name = [
            'MAV', 'RMS', 'VAR', 'STD', 'ZC', 'MPR',
            'MED', 'P10', 'P90', 'IQR',
            'SKEW', 'KURT',
            'SLOPE', 'INTERCEPT', 'R2',
            'AC1', 'NPEAKS',
            'SPEC_CENT', 'SPEC_ENT'
        ]

    
    def fit(self, X, y=None):
        _, self.n_channels_, _ = X.shape
        return self

    def transform(self, X):
        self.sigma = np.std(X)
        windows, channels, window_length = X.shape  #window_length nécessaire ?
        features_number = len(self.features_name)
        transformed = np.zeros((windows, channels * features_number))

        self.sigma = np.std(X)
        t = np.arange(window_length)
        
        for window in range(windows):
            for channel in range(channels):
                features = self.compute_features(X[window, channel, :])
                transformed[window, channel*features_number:(channel+1)*features_number] = features
        return transformed


    def get_feature_names_out(self, input_features=None):
        names = []
        for ch in range(self.n_channels_):
            for fn in self.features_name:
                names.append(f"ch{ch}_{fn}")
        return np.array(names)

    def compute_features(self, x):

        t = np.arange(len(x))
 
        mav = np.mean(np.abs(x))
        rms = np.sqrt(np.mean(x**2))
        var = np.var(x, ddof=1)
        std = np.std(x, ddof=1)
        zc = np.sum(np.diff(np.sign(x)) != 0)
        mpr = np.sum(np.abs(x) > self.sigma) / len(x)


        med = np.median(x)
        p10 = np.percentile(x, 10)
        p90 = np.percentile(x, 90)
        iqr = p90 - p10
        sk = skew(x)
        kt = kurtosis(x)


        coef = np.polyfit(t, x, 1)
        slope, intercept = coef
        corr = np.corrcoef(t, x)[0, 1]
        r2 = corr**2 if not np.isnan(corr) else 0.0


        ac1 = np.corrcoef(x[:-1], x[1:])[0, 1] if len(x) > 1 else 0.0


        peaks, _ = find_peaks(x)
        n_peaks = len(peaks)

        Xf = np.abs(rfft(x))
        freqs = rfftfreq(len(x), 1.0)  # assume unit sampling
        spec_cent = (freqs * Xf).sum() / (Xf.sum() + 1e-12)
        p = Xf / (Xf.sum() + 1e-12)
        spec_ent = -np.sum(p * np.log2(p + 1e-12))

        return [
            mav, rms, var, std, zc, mpr,
            med, p10, p90, iqr,
            sk, kt,
            slope, intercept, r2,
            ac1, n_peaks,
            spec_cent, spec_ent
        ]


            

In [40]:
 
from sklearn.feature_selection import RFE

 
 

pipe = Pipeline([                                                               #We then start to define our pipeline
    ('feat',   FeatureExtractor()),                                             #We initiate our custom feature extraction class that we defined earlier 
    ('select', RFE(estimator=RandomForestRegressor(
            random_state=42, n_jobs=-1),n_features_to_select=10)),   
    ('scale', StandardScaler()),                                                #As different features we will extract are on different scale (ex: enthropy, RMS), we scale them
    ('model',  'passthrough'),                                                  #We start with a model (doesn't really matter which one as they will be selected in our param_grid) (maybe it can be removed ?)
                                                  
])

param_grid = [                                                                  #We define our param_grid with the different models we will use and the hyperparamaeters for each to try. (Maybe a better selection of model would be better) (Explain the revelancy of each models choosen)

  {
    'select__n_features_to_select': [5, 10, 20],
    'model': [DecisionTreeRegressor(random_state=42)],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 3, 5],
  },

  {
    'select__n_features_to_select': [5, 10, 20],
    'model': [RandomForestRegressor(random_state=42, n_jobs=4)],
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [10, 20],
    'model__max_features': ['sqrt', 'log2'],
  },


    {
    'select__n_features_to_select': [5, 10, 20],
    'model': [MultiOutputRegressor(SVR(kernel='rbf'))],
    'model__estimator__C': [0.1, 1.0],
    'model__estimator__epsilon': [0.1, 0.2]
  },



    {
    'select__n_features_to_select': [5, 10, 20],
    'model': [HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1)],
    'model__max_iter': [100, 200],
    'model__learning_rate': [0.01, 0.1]
    }
]


#gkf = GroupKFold(n_splits=5)                             #We do the group folding 
gkf = logo
rmse  = make_scorer(mean_squared_error, greater_is_better=False, squared=False)                    #We state the score parameter we choose 

gs = GridSearchCV(                                       #Here the GridSearch function will check for the best model with the best combination of features (still have to integrate the (RFE ?) wrapper)  and hyperparameters
    pipe,
    param_grid,
    cv=gkf,
    scoring=rmse,
    n_jobs=4,
    verbose=2
)


gs.fit(X, Y, groups=groups)                                  #We do the fitting



results = pd.DataFrame(gs.cv_results_)
results['RMSE'] = -results['mean_test_score']
results = results.sort_values('RMSE')
print(results[['params', 'RMSE']].to_string(index=False))
#As computation for every model is rather annoying, we will pick only 2 



Fitting 5 folds for each of 87 candidates, totalling 435 fits


KeyboardInterrupt: 